# NB12 — Stress Testing & Final Report

**Objectives**:
1. **Historical Stress Scenarios**: Replay the optimized portfolio through 5 known crises
2. **Hypothetical Stress Scenarios**: Taiwan crisis, AI bubble burst, rate shock, cyber breach
3. **Monte Carlo Simulation**: 10,000 paths using calibrated GARCH vols + DCC correlations
4. **Sensitivity Analysis**: Factor exposure decomposition
5. **Final Summary Dashboard**: Executive summary of all 11 notebooks

**Output**: `stress_test_results.csv`, `monte_carlo_distribution.png`, figures

In [1]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.config import *
from src.backtest_engine import (
    compute_all_metrics, annualized_return, annualized_volatility,
    max_drawdown_from_returns, sharpe_ratio
)
from src.portfolio_optimizer import covariance_ledoit_wolf
from src.visualization import save_fig

print('Imports OK')

Imports OK


## 1. Load Portfolio & Data

In [2]:
# ── Load data ──
master = pd.read_parquet(MASTER_DATA_FILE)
avail_tickers = [t for t in TICKERS if t in master.columns]
prices = master[avail_tickers].dropna(how='all')
returns = prices.pct_change().dropna(how='all')  # keep rows where at least 1 ticker has data
log_returns = np.log(prices / prices.shift(1)).dropna(how='all')

# ── Load portfolio weights from NB11 ──
if PORTFOLIO_WEIGHTS_FILE.exists():
    portfolio_weights = pd.read_parquet(PORTFOLIO_WEIGHTS_FILE)
    # Use latest weights as the "current" portfolio
    latest_weights = portfolio_weights.iloc[-1]
    w = latest_weights[avail_tickers].values if all(t in latest_weights.index for t in avail_tickers) else np.ones(len(avail_tickers)) / len(avail_tickers)
    print(f'Loaded portfolio weights from NB11')
else:
    # Fallback: equal weight
    w = np.ones(len(avail_tickers)) / len(avail_tickers)
    print('Using equal-weight portfolio (NB11 weights not available)')

# ── Load backtest performance ──
if BACKTEST_PERF_FILE.exists():
    backtest_perf = pd.read_csv(BACKTEST_PERF_FILE, index_col=0)
    print(f'Loaded backtest performance: {len(backtest_perf)} strategies')
else:
    backtest_perf = None

# ── Load conditional volatility (NB03) for Monte Carlo ──
cond_vol = pd.read_parquet(COND_VOL_FILE) if COND_VOL_FILE.exists() else None

# ── Load GARCH params (NB03) ──
garch_params = pd.read_csv(GARCH_PARAMS_FILE) if GARCH_PARAMS_FILE.exists() else None

print(f'\nPortfolio: {len(avail_tickers)} assets')
print(f'Top 5 weights: {dict(sorted(zip(avail_tickers, w), key=lambda x: -x[1])[:5])}')


Using equal-weight portfolio (NB11 weights not available)

Portfolio: 20 assets
Top 5 weights: {'NVDA': np.float64(0.05), 'AVGO': np.float64(0.05), 'TSM': np.float64(0.05), 'SNPS': np.float64(0.05), 'MSFT': np.float64(0.05)}


## 2. Historical Stress Scenarios

Replay the portfolio through 5 known crisis periods. For each:
- Portfolio return, max drawdown, worst single-day loss
- Comparison with SPY and XLK benchmarks
- Identify which assets drove the losses

In [3]:
# ── Define historical stress scenarios ──
stress_scenarios = {
    'COVID Crash': ('2020-02-19', '2020-03-23'),
    'Rate Shock 2022': ('2022-01-03', '2022-10-13'),
    'SVB Contagion': ('2023-03-08', '2023-03-15'),
    'Chip Export Ban': ('2022-10-07', '2022-11-07'),
    'DeepSeek/Tariff Shock': ('2025-01-27', '2025-02-10'),
}

stress_results = []

for event_name, (start, end) in stress_scenarios.items():
    mask = (returns.index >= start) & (returns.index <= end)
    event_returns = returns.loc[mask, avail_tickers]
    
    if len(event_returns) == 0:
        print(f'{event_name}: No data in period')
        continue
    
    # Fill NaN for tickers not yet listed (pre-IPO) with 0 return
    event_returns = event_returns.fillna(0)
    
    # Portfolio return during stress
    port_daily = event_returns.values @ w
    port_cumulative = (1 + pd.Series(port_daily)).prod() - 1
    port_maxdd = max_drawdown_from_returns(pd.Series(port_daily))
    worst_day = pd.Series(port_daily).min()
    
    # Per-asset contribution to loss
    asset_cum_returns = (1 + event_returns).prod() - 1
    weighted_contribution = asset_cum_returns * w
    worst_assets = weighted_contribution.nsmallest(3)
    
    # Benchmark comparison
    spy_ret = np.nan
    if 'BM_SPY' in master.columns:
        spy_rets = master['BM_SPY'].pct_change().loc[mask]
        if len(spy_rets) > 0:
            spy_ret = (1 + spy_rets).prod() - 1
    
    result = {
        'scenario': event_name,
        'start': start,
        'end': end,
        'n_days': len(event_returns),
        'portfolio_return': port_cumulative,
        'max_drawdown': port_maxdd,
        'worst_day': worst_day,
        'spy_return': spy_ret,
        'relative_perf': port_cumulative - spy_ret if not np.isnan(spy_ret) else np.nan,
        'worst_contributor_1': f'{worst_assets.index[0]} ({worst_assets.iloc[0]:.2%})',
        'worst_contributor_2': f'{worst_assets.index[1]} ({worst_assets.iloc[1]:.2%})' if len(worst_assets) > 1 else '',
        'worst_contributor_3': f'{worst_assets.index[2]} ({worst_assets.iloc[2]:.2%})' if len(worst_assets) > 2 else '',
    }
    stress_results.append(result)
    
    print(f'\n{event_name} ({start} → {end}, {len(event_returns)}d):')
    print(f'  Portfolio: {port_cumulative:.2%} | MaxDD: {port_maxdd:.2%} | Worst day: {worst_day:.2%}')
    print(f'  SPY: {spy_ret:.2%}' if not np.isnan(spy_ret) else '  SPY: N/A')
    print(f'  Worst contributors: {worst_assets.index[0]} ({worst_assets.iloc[0]:.2%})')

stress_df = pd.DataFrame(stress_results)
stress_df


COVID Crash: No data in period
Rate Shock 2022: No data in period
SVB Contagion: No data in period
Chip Export Ban: No data in period

DeepSeek/Tariff Shock (2025-01-27 → 2025-02-10, 11d):
  Portfolio: 0.66% | MaxDD: -1.68% | Worst day: -4.88%
  SPY: N/A
  Worst contributors: AMD (-0.50%)


,scenario,start,end,n_days,portfolio_return,max_drawdown,worst_day,spy_return,relative_perf,worst_contributor_1,worst_contributor_2,worst_contributor_3
0,DeepSeek/Tariff Shock,2025-01-27,2025-02-10,11,0.006643,-0.016772,-0.048793,NaN,NaN,AMD (-0.50%),NOW (-0.46%),MSFT (-0.36%)


## 3. Hypothetical Stress Scenarios

Apply user-defined shocks to specific assets and compute portfolio impact.
These scenarios are not observed in history but represent plausible tail events.

In [4]:
# ── Define hypothetical scenarios ──
# Each scenario specifies the shock (%) to each affected ticker
hypothetical_scenarios = {
    'Taiwan Strait Crisis': {
        'TSM': -0.50, 'NVDA': -0.30, 'AAPL': -0.25, 'AVGO': -0.20,
        'AMD': -0.25, 'MU': -0.20, 'SNPS': -0.15,
        # Spillover to broader tech
        'MSFT': -0.10, 'AMZN': -0.10, 'META': -0.10, 'GOOG': -0.10,
    },
    'AI Bubble Burst': {
        'NVDA': -0.40, 'PLTR': -0.40, 'CRWD': -0.30, 'DDOG': -0.30,
        'AMD': -0.35, 'AVGO': -0.25, 'ANET': -0.30,
        'MSFT': -0.15, 'META': -0.20, 'GOOG': -0.15, 'AMZN': -0.15,
        # Defensive names less affected
        'AAPL': -0.10, 'SAP': -0.05, 'CRM': -0.15, 'NOW': -0.15,
    },
    'Fed Emergency Rate Hike (+100bps)': {
        # Duration-sensitive high-growth names hit hardest
        'PLTR': -0.25, 'CRWD': -0.20, 'DDOG': -0.20, 'NOW': -0.15,
        'NVDA': -0.15, 'CRM': -0.12,
        # Mature names less affected
        'AAPL': -0.05, 'MSFT': -0.08, 'GOOG': -0.08, 'SAP': -0.05,
        # XYZ affected by rate-sensitive consumer spend
        'XYZ': -0.20,
    },
    'Major Cybersecurity Breach': {
        # Cyber stocks rally (increased demand)
        'PANW': 0.15, 'CRWD': 0.15, 'DDOG': 0.10,
        # Breached platform companies fall
        'META': -0.10, 'GOOG': -0.10, 'AMZN': -0.08,
        'MSFT': -0.05,
    },
}

hypo_results = []

for scenario_name, shocks in hypothetical_scenarios.items():
    # Build shock vector for available tickers
    shock_vec = np.zeros(len(avail_tickers))
    for i, t in enumerate(avail_tickers):
        if t in shocks:
            shock_vec[i] = shocks[t]
    
    # Portfolio impact = w' * shock
    portfolio_impact = w @ shock_vec
    
    # Individual contributions
    contributions = pd.Series(w * shock_vec, index=avail_tickers)
    
    hypo_results.append({
        'scenario': scenario_name,
        'portfolio_impact': portfolio_impact,
        'n_assets_shocked': (shock_vec != 0).sum(),
        'worst_contributor': contributions.idxmin(),
        'worst_contribution': contributions.min(),
        'best_contributor': contributions.idxmax() if contributions.max() > 0 else 'None',
        'best_contribution': contributions.max(),
    })
    
    print(f'\n{scenario_name}:')
    print(f'  Portfolio impact: {portfolio_impact:.2%}')
    print(f'  Worst: {contributions.idxmin()} ({contributions.min():.2%} contribution)')
    if contributions.max() > 0:
        print(f'  Best:  {contributions.idxmax()} ({contributions.max():.2%} contribution)')

hypo_df = pd.DataFrame(hypo_results)

# ── Visualization ──
fig, ax = plt.subplots(figsize=(12, 6))
colors = ['red' if x < 0 else 'green' for x in hypo_df['portfolio_impact']]
bars = ax.barh(hypo_df['scenario'], hypo_df['portfolio_impact'] * 100, color=colors, alpha=0.7)
ax.set_xlabel('Portfolio Impact (%)')
ax.set_title('Hypothetical Stress Scenario Impact')
ax.axvline(x=0, color='black', linewidth=0.5)
for bar, val in zip(bars, hypo_df['portfolio_impact']):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1%}', va='center', fontsize=10)
save_fig(fig, 'nb12_hypothetical_stress')
plt.show()


Taiwan Strait Crisis:
  Portfolio impact: -11.25%
  Worst: TSM (-2.50% contribution)

AI Bubble Burst:
  Portfolio impact: -17.00%
  Worst: NVDA (-2.00% contribution)

Fed Emergency Rate Hike (+100bps):
  Portfolio impact: -7.65%
  Worst: PLTR (-1.25% contribution)

Major Cybersecurity Breach:
  Portfolio impact: 0.35%
  Worst: META (-0.50% contribution)
  Best:  PANW (0.75% contribution)


## 4. Monte Carlo Simulation

Generate 10,000 return paths using the calibrated covariance matrix.
For each path, compute the portfolio return and track:
- VaR and CVaR distributions
- Probability of drawdown > 20% over 1-year horizon

**Method**: Multivariate normal with Cholesky decomposition of the
annualized covariance matrix (or use empirical distribution via bootstrap).

$$R_t \sim \mathcal{N}(\mu, \Sigma)$$
$$R_{sim} = \mu_{daily} + L \cdot Z, \quad Z \sim \mathcal{N}(0, I)$$

where $L$ is the Cholesky factor of the daily covariance matrix.

In [ ]:
# ── Monte Carlo parameters ──
n_paths = MC_PATHS  # 10,000
horizon_days = 252  # 1-year

# ── Calibrate from historical data ──
# Note: This uses constant covariance (sample estimate). For GARCH-recursive
# Monte Carlo with time-varying vol, integrate cond_vol from NB03's GARCH models.
# The constant-cov approach underestimates tail risk due to missing vol clustering.
# Use last 504 days (2 years) for parameter estimation
recent_returns = returns[avail_tickers].tail(504).dropna()
mu_daily = recent_returns.mean().values
cov_daily = recent_returns.cov().values

# Ensure positive semi-definite (numerical fix)
eigvals = np.linalg.eigvalsh(cov_daily)
if eigvals.min() < 0:
    cov_daily += np.eye(len(avail_tickers)) * (abs(eigvals.min()) + 1e-10)

# Cholesky decomposition
L = np.linalg.cholesky(cov_daily)

# ── Simulate ──
np.random.seed(RANDOM_STATE)

# Terminal portfolio values
terminal_returns = np.zeros(n_paths)
max_drawdowns = np.zeros(n_paths)
annual_vols = np.zeros(n_paths)

for path in range(n_paths):
    # Generate correlated daily returns
    Z = np.random.randn(horizon_days, len(avail_tickers))
    sim_returns = mu_daily + Z @ L.T  # (252, N)
    
    # Portfolio daily returns
    port_daily = sim_returns @ w  # (252,)
    
    # Cumulative return
    cum_return = np.prod(1 + port_daily) - 1
    terminal_returns[path] = cum_return
    
    # Max drawdown in this path
    cumulative = np.cumprod(1 + port_daily)
    running_max = np.maximum.accumulate(cumulative)
    drawdowns = cumulative / running_max - 1
    max_drawdowns[path] = drawdowns.min()
    
    # Annualized vol
    annual_vols[path] = np.std(port_daily) * np.sqrt(252)

# ── Results ──
sep = '=' * 50
print(f'Monte Carlo Simulation ({n_paths:,} paths, {horizon_days} days)')
print(sep)
print(f'Expected Annual Return: {np.mean(terminal_returns):.2%}')
print(f'Return Std Dev:         {np.std(terminal_returns):.2%}')
print(f'Median Return:          {np.median(terminal_returns):.2%}')
print()
print(f'VaR (95%):  {np.percentile(terminal_returns, 5):.2%}')
print(f'VaR (99%):  {np.percentile(terminal_returns, 1):.2%}')
print(f'CVaR (95%): {terminal_returns[terminal_returns <= np.percentile(terminal_returns, 5)].mean():.2%}')
print(f'CVaR (99%): {terminal_returns[terminal_returns <= np.percentile(terminal_returns, 1)].mean():.2%}')
print()
print(f'P(Drawdown > 20%): {(max_drawdowns < -0.20).mean():.1%}')
print(f'P(Drawdown > 30%): {(max_drawdowns < -0.30).mean():.1%}')
print(f'P(Drawdown > 40%): {(max_drawdowns < -0.40).mean():.1%}')
print(f'Expected Max DD:   {np.mean(max_drawdowns):.2%}')
print(f'Worst-case DD (99th pct): {np.percentile(max_drawdowns, 1):.2%}')


In [ ]:
# ── Monte Carlo Distribution Plots ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel A: Terminal return distribution
ax = axes[0]
ax.hist(terminal_returns * 100, bins=100, density=True, alpha=0.7, color='steelblue', edgecolor='white')
var_95 = np.percentile(terminal_returns, 5) * 100
var_99 = np.percentile(terminal_returns, 1) * 100
ax.axvline(x=var_95, color='orange', linestyle='--', label=f'VaR 95% = {var_95:.1f}%')
ax.axvline(x=var_99, color='red', linestyle='--', label=f'VaR 99% = {var_99:.1f}%')
ax.axvline(x=np.mean(terminal_returns) * 100, color='green', linestyle='-', label=f'Mean = {np.mean(terminal_returns)*100:.1f}%')
ax.set_xlabel('1-Year Return (%)')
ax.set_ylabel('Density')
ax.set_title('MC: 1-Year Return Distribution')
ax.legend(fontsize=8)

# Panel B: Max drawdown distribution
ax = axes[1]
ax.hist(max_drawdowns * 100, bins=100, density=True, alpha=0.7, color='salmon', edgecolor='white')
ax.axvline(x=-20, color='orange', linestyle='--', label='DD = -20%')
ax.axvline(x=-30, color='red', linestyle='--', label='DD = -30%')
ax.set_xlabel('Max Drawdown (%)')
ax.set_ylabel('Density')
ax.set_title('MC: Max Drawdown Distribution')
ax.legend(fontsize=8)

# Panel C: Return vs Drawdown scatter
ax = axes[2]
sample_idx = np.random.choice(n_paths, min(2000, n_paths), replace=False)
ax.scatter(max_drawdowns[sample_idx] * 100, terminal_returns[sample_idx] * 100,
           alpha=0.2, s=5, c='steelblue')
ax.set_xlabel('Max Drawdown (%)')
ax.set_ylabel('1-Year Return (%)')
ax.set_title('Return vs Max Drawdown')
ax.axhline(y=0, color='black', linewidth=0.5)
ax.axvline(x=-20, color='red', linewidth=0.5, linestyle='--')

save_fig(fig, 'nb12_monte_carlo_distributions')
plt.show()

## 5. Sensitivity Analysis

Measure portfolio P&L impact per +1 sigma move in key risk factors:
- VIX (implied volatility)
- DXY (US dollar index)
- 10Y yield
- SPY (broad market)

This is a linear factor exposure decomposition using multivariate regression:
$R_{port,t} = \alpha + \sum_k \beta_k \cdot F_{k,t} + \varepsilon_t$

In [ ]:
import statsmodels.api as sm

# ── Compute portfolio returns ──
port_returns = returns[avail_tickers].values @ w
port_returns = pd.Series(port_returns, index=returns.index, name='Portfolio')

# ── Load factor returns ──
factors = pd.DataFrame(index=returns.index)

factor_tickers = {
    'SPY': 'BM_SPY',
    'VIX_change': '^VIX',
    'TLT': 'TLT',
    'GLD': 'GLD',
    'DXY': 'DX-Y.NYB',
}

for fname, ticker in factor_tickers.items():
    if ticker in master.columns:
        factors[fname] = master[ticker].pct_change()

# Align
aligned = pd.concat([port_returns, factors], axis=1).dropna()
if len(aligned) > 100 and len(aligned.columns) > 1:
    y = aligned['Portfolio']
    X = aligned.drop('Portfolio', axis=1)
    X = sm.add_constant(X)
    
    # OLS regression with Newey-West HAC standard errors
    model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 5})
    
    print('── Factor Exposure Decomposition ──')
    print(f'R² = {model.rsquared:.4f} (fraction of portfolio variance explained by factors)\n')
    
    factor_summary = pd.DataFrame({
        'Beta': model.params[1:],  # exclude constant
        't-stat': model.tvalues[1:],
        'p-value': model.pvalues[1:],
    })
    
    # Impact of +1 sigma move in each factor
    factor_stds = X.iloc[:, 1:].std()  # daily std of each factor
    factor_summary['Impact (+1σ daily)'] = model.params[1:] * factor_stds
    factor_summary['Impact (+1σ annual)'] = factor_summary['Impact (+1σ daily)'] * np.sqrt(252)
    
    print(factor_summary.to_string())
    
    # ── Residual analysis ──
    residual_vol = model.resid.std() * np.sqrt(252)
    total_vol = y.std() * np.sqrt(252)
    systematic_vol = np.sqrt(total_vol**2 - residual_vol**2) if total_vol > residual_vol else 0
    
    print(f'\n── Risk Decomposition ──')
    print(f'Total portfolio vol:      {total_vol:.2%}')
    print(f'Systematic (factor) vol:  {systematic_vol:.2%} ({systematic_vol/total_vol*100:.1f}%)')
    print(f'Idiosyncratic vol:        {residual_vol:.2%} ({residual_vol/total_vol*100:.1f}%)')
else:
    print('Insufficient factor data for sensitivity analysis')


## 6. Per-Ticker Risk Card

Summary risk card for each ticker: annualized vol, VaR, max drawdown,
beta, current weight, and risk contribution.

In [ ]:
# ── Risk cards ──
risk_cards = []
cov_annual = covariance_ledoit_wolf(returns[avail_tickers])
port_vol_total = np.sqrt(w @ cov_annual @ w)

for i, ticker in enumerate(avail_tickers):
    r = returns[ticker].dropna()
    
    # Annualized volatility
    ann_vol = r.std() * np.sqrt(252)
    
    # VaR 95% (historical)
    var_95 = np.percentile(r, 5)
    
    # CVaR 95%
    cvar_95 = r[r <= var_95].mean()
    
    # Max drawdown
    mdd = max_drawdown_from_returns(r)
    
    # Beta to SPY
    beta = np.nan
    if 'BM_SPY' in master.columns:
        spy_r = master['BM_SPY'].pct_change().reindex(r.index).dropna()
        common = r.index.intersection(spy_r.index)
        if len(common) > 60:
            cov_spy = np.cov(r.loc[common], spy_r.loc[common])
            beta = cov_spy[0, 1] / cov_spy[1, 1]
    
    # Marginal risk contribution
    # RC_i = w_i * (Sigma @ w)_i / sigma_p
    marginal = cov_annual @ w
    risk_contribution = w[i] * marginal[i] / port_vol_total
    pct_risk = risk_contribution / port_vol_total * 100
    
    risk_cards.append({
        'ticker': ticker,
        'weight': w[i],
        'ann_vol': ann_vol,
        'var_95': var_95,
        'cvar_95': cvar_95,
        'max_dd': mdd,
        'beta_spy': beta,
        'risk_contribution': risk_contribution,
        'pct_of_risk': pct_risk,
    })

risk_df = pd.DataFrame(risk_cards).set_index('ticker')

# Format for display
print('── Per-Ticker Risk Cards ──')
display = risk_df.copy()
for col in ['weight', 'ann_vol', 'var_95', 'cvar_95', 'max_dd', 'risk_contribution']:
    display[col] = display[col].map('{:.2%}'.format)
display['beta_spy'] = display['beta_spy'].map('{:.2f}'.format)
display['pct_of_risk'] = display['pct_of_risk'].map('{:.1f}%'.format)
print(display.to_string())


## 6b. Advanced Stress Dashboard Metrics

- **Ulcer Index** per stress scenario (sustained drawdown severity)
- **CDaR at 95%** per stress scenario  
- **CDB** (Conditional Diversification Benefit) under each stress scenario
- **Factor variance decomposition**: market vs. sector vs. idiosyncratic

In [ ]:
from src.backtest_engine import ulcer_index, pain_index, conditional_drawdown_at_risk
from src.portfolio_optimizer import conditional_diversification_benefit

# ── Ulcer Index & CDaR per stress scenario ──
print('── Advanced Stress Metrics ──')
advanced_stress = []
for event_name, (start, end) in stress_scenarios.items():
    mask = (returns.index >= start) & (returns.index <= end)
    event_returns = returns.loc[mask, avail_tickers]
    if len(event_returns) == 0:
        continue
    
    port_daily = event_returns.values @ w
    
    ui = ulcer_index(port_daily, is_returns=True)
    pi = pain_index(port_daily, is_returns=True)
    cdar = conditional_drawdown_at_risk(port_daily, alpha=0.05) if len(port_daily) > 10 else np.nan
    
    # CDB under stress
    stress_cov = event_returns.cov().values * 252
    cdb = conditional_diversification_benefit(w, stress_cov)
    
    advanced_stress.append({
        'scenario': event_name, 'ulcer_index': ui, 'pain_index': pi,
        'cdar_95': cdar, 'cdb': cdb
    })
    print(f'  {event_name}: Ulcer={ui:.4f}, Pain={pi:.4f}, CDaR95={cdar:.4f}, CDB={cdb:.4f}')

adv_stress_df = pd.DataFrame(advanced_stress)

# ── Factor Variance Decomposition ──
print('\n── Factor Variance Decomposition ──')
if 'model' in dir() and hasattr(model, 'rsquared'):
    total_var = port_returns.var() * 252
    systematic_var = total_var * model.rsquared
    idiosyncratic_var = total_var * (1 - model.rsquared)
    
    # Per-factor contribution
    print(f'  Total annualized variance:  {total_var:.6f}')
    print(f'  Systematic (R²={model.rsquared:.2%}): {systematic_var:.6f}')
    print(f'  Idiosyncratic:             {idiosyncratic_var:.6f}')
    
    # Approximate per-factor contribution via beta^2 * var(factor)
    factor_cols = [c for c in model.params.index if c != 'const']
    factor_returns_aligned = factors.reindex(port_returns.index).dropna()
    for fc in factor_cols:
        if fc in model.params and fc in factor_returns_aligned.columns:
            beta_sq = model.params[fc] ** 2
            fvar = factor_returns_aligned[fc].var() * 252
            contribution = beta_sq * fvar
            pct = contribution / total_var * 100
            print(f'    {fc:15s}: β²·Var(f) = {contribution:.6f} ({pct:.1f}%)')
else:
    print('  Factor model not available — run sensitivity analysis first')

# ── Full-sample CDB ──
cdb_full = conditional_diversification_benefit(w, cov_annual)
print(f'\n  Full-sample CDB: {cdb_full:.4f}')
print('  (CDB close to 0 → high correlation, close to 1 → maximum diversification)')

## 7. Risk Contribution Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Panel A: Weight vs Risk Contribution
ax = axes[0]
x = np.arange(len(avail_tickers))
width = 0.35
ax.bar(x - width/2, risk_df['weight'] * 100, width, label='Weight (%)', color='steelblue')
ax.bar(x + width/2, risk_df['pct_of_risk'], width, label='Risk Contribution (%)', color='salmon')
ax.set_xticks(x)
ax.set_xticklabels(avail_tickers, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('%')
ax.set_title('Weight vs Risk Contribution')
ax.legend()

# Panel B: Risk contribution pie chart
ax = axes[1]
# Group by sector for cleaner visualization
sector_risk = {}
for group_name, group_tickers in SECTOR_GROUPS.items():
    sector_indices = [avail_tickers.index(t) for t in group_tickers if t in avail_tickers]
    if sector_indices:
        sector_risk[group_name] = risk_df.iloc[sector_indices]['pct_of_risk'].sum()

if sector_risk:
    labels = list(sector_risk.keys())
    sizes = list(sector_risk.values())
    ax.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90)
    ax.set_title('Risk Contribution by Sector')

save_fig(fig, 'nb12_risk_contribution')
plt.show()

## 8. Executive Summary Dashboard

Consolidated findings from the entire 12-notebook pipeline.

In [ ]:
print('='*70)
print('EXECUTIVE SUMMARY — Tech Sector Risk Management Pipeline')
print('='*70)

print(f'\n1. UNIVERSE: {len(avail_tickers)} tech stocks across '
      f'{len(SECTOR_GROUPS)} sub-sectors')
print(f'   Date range: {returns.index[0].strftime("%Y-%m-%d")} → {returns.index[-1].strftime("%Y-%m-%d")}')

print(f'\n2. PORTFOLIO CONSTRUCTION:')
print(f'   Optimization methods tested: 5 (MV, CVaR, BL, HRP, ERC)')
print(f'   Constraints: {MAX_SINGLE_STOCK_WEIGHT:.0%} max stock, '
      f'{MAX_SECTOR_WEIGHT:.0%} max sector, long-only')

if backtest_perf is not None:
    print(f'\n3. BACKTEST RESULTS:')
    for idx, row in backtest_perf.iterrows():
        print(f'   {idx}: Sharpe={row.get("sharpe_ratio", "N/A"):.2f}, '
              f'Return={row.get("annualized_return", "N/A"):.2%}, '
              f'MaxDD={row.get("max_drawdown", "N/A"):.2%}')

print(f'\n4. STRESS TEST SUMMARY:')
for _, row in stress_df.iterrows():
    print(f'   {row["scenario"]}: {row["portfolio_return"]:.2%}')

print(f'\n5. MONTE CARLO (1-Year, {n_paths:,} paths):')
print(f'   Expected return: {np.mean(terminal_returns):.2%}')
print(f'   VaR 95%: {np.percentile(terminal_returns, 5):.2%}')
print(f'   CVaR 95%: {terminal_returns[terminal_returns <= np.percentile(terminal_returns, 5)].mean():.2%}')
print(f'   P(DD > 20%): {(max_drawdowns < -0.20).mean():.1%}')

print(f'\n6. HYPOTHETICAL STRESS SCENARIOS:')
for _, row in hypo_df.iterrows():
    print(f'   {row["scenario"]}: {row["portfolio_impact"]:.2%}')

print(f'\n7. KEY RISK FACTORS:')
if 'factor_summary' in dir():
    significant = factor_summary[factor_summary['p-value'] < 0.05]
    for idx, row in significant.iterrows():
        print(f'   {idx}: beta={row["Beta"]:.3f} (p={row["p-value"]:.4f})')

## 9. Save All Outputs

In [ ]:
# ── Save stress test results ──
# Save historical and hypothetical results separately (different schemas)
if len(stress_df) > 0:
    stress_df.to_csv(TABLES_DIR / 'historical_stress_results.csv', index=False)
    print(f'Saved historical stress: {TABLES_DIR / "historical_stress_results.csv"}')
if len(hypo_df) > 0:
    hypo_df.to_csv(TABLES_DIR / 'hypothetical_stress_results.csv', index=False)
    print(f'Saved hypothetical stress: {TABLES_DIR / "hypothetical_stress_results.csv"}')

# Combined summary with shared columns only
shared_cols = ['scenario']
stress_summary = pd.concat([
    stress_df[['scenario', 'portfolio_return']].rename(columns={'portfolio_return': 'impact'}) if len(stress_df) > 0 else pd.DataFrame(),
    hypo_df[['scenario', 'portfolio_impact']].rename(columns={'portfolio_impact': 'impact'}) if len(hypo_df) > 0 else pd.DataFrame(),
], ignore_index=True)
all_stress = stress_summary
all_stress.to_csv(STRESS_TEST_FILE, index=False)
print(f'Saved: {STRESS_TEST_FILE}')

# ── Save Monte Carlo summary ──
mc_summary = pd.DataFrame({
    'metric': [
        'Expected Return', 'Return Std', 'Median Return',
        'VaR 95%', 'VaR 99%', 'CVaR 95%', 'CVaR 99%',
        'P(DD>20%)', 'P(DD>30%)', 'Expected MaxDD',
    ],
    'value': [
        np.mean(terminal_returns), np.std(terminal_returns), np.median(terminal_returns),
        np.percentile(terminal_returns, 5), np.percentile(terminal_returns, 1),
        terminal_returns[terminal_returns <= np.percentile(terminal_returns, 5)].mean(),
        terminal_returns[terminal_returns <= np.percentile(terminal_returns, 1)].mean(),
        (max_drawdowns < -0.20).mean(), (max_drawdowns < -0.30).mean(),
        np.mean(max_drawdowns),
    ]
})
mc_file = TABLES_DIR / 'monte_carlo_summary.csv'
mc_summary.to_csv(mc_file, index=False)
print(f'Saved: {mc_file}')

# ── Save risk cards ──
risk_cards_file = TABLES_DIR / 'risk_cards.csv'
risk_df.to_csv(risk_cards_file)
print(f'Saved: {risk_cards_file}')

print('\n' + '='*60)
print('NB12 COMPLETE — All Outputs Generated')
print('='*60)
